# Silver: tablas historicas en Iceberg con MERGE (respaldo manual de Airflow)

Hace lo mismo que la tarea de Silver del DAG: lee todos los anios de Bronze, cura las columnas, agrega `fecha_cargue` y hace `MERGE INTO` en `nessie.silver.<tabla>` (bucket `silver`).

Reutiliza las funciones de `dags/scripts/silver_transform.py`.

**Antes de correr:** Bronze cargado en MinIO.

- `MODO = "local"`: Spark corre dentro de Jupyter (apaga el clúster para ahorrar RAM: `docker compose stop spark-master spark-worker`).
- `MODO = "cluster"`: usa el clúster Spark (`docker compose up -d spark-master spark-worker`); es la alternativa si el modo local se cae por memoria.

In [ ]:
import os
import socket
import sys
from pathlib import Path

# Los scripts viven en dags/scripts (la misma carpeta que usa Airflow)
for candidato in [Path("../dags/scripts"), Path("/workspace/dags/scripts"), Path("dags/scripts")]:
    if candidato.exists():
        sys.path.insert(0, str(candidato.resolve()))
        break

from pyspark.sql import functions as F
import silver_transform as silver

## Parametros

- Pipeline de Airflow: `users` (users_hist)
- Manual: `post` (post_hist), une post_2020 y post_2021

`MAX_ROWS` limita los registros (el enunciado lo permite); `None` = sin limite.

In [ ]:
DATASET = "users"   # "users" o "post"
MAX_ROWS = 200000   # None = sin limite (requiere mas RAM)
MODO = "local"    # "local" o "cluster"

## Sesion de Spark

El catalogo `nessie` y la conexion a MinIO salen del `spark-defaults.conf` de la imagen. Si cambias `MODO`, reinicia el kernel antes de crear la sesion.

In [ ]:
# Mismas opciones de JVM que el spark-defaults.conf (al pasar --driver-java-options se reemplazan)
JAVA_OPTS = "-Duser.timezone=UTC -Daws.region=us-east-1"

if MODO == "local":
    # 1 hilo y 2 GB; compilador C1 y GC Serial = JVM mas conservadora
    args = ("--master local[1] --driver-memory 2g "
            f'--driver-java-options "{JAVA_OPTS} -XX:TieredStopAtLevel=1 -XX:+UseSerialGC" pyspark-shell')
else:
    ip = socket.gethostbyname(socket.gethostname())  # para que los executors encuentren a Jupyter
    args = ("--master spark://spark-master:7077 --driver-memory 1g "
            f"--conf spark.driver.host={ip} --conf spark.executor.memory=512m --conf spark.cores.max=1 "
            f'--driver-java-options "{JAVA_OPTS}" pyspark-shell')
os.environ["PYSPARK_SUBMIT_ARGS"] = args

cfg = silver.DATASETS[DATASET]
spark = silver.get_spark(f"silver_{cfg['table']}")
spark

## Lectura de Bronze (todos los anios) y transformacion

In [ ]:
raw = spark.read.parquet(cfg["source"])
print("Origen:", cfg["source"])
raw.printSchema()

In [ ]:
df = cfg["transform"](raw).withColumn("fecha_cargue", F.current_timestamp())
if MAX_ROWS:
    df = df.limit(int(MAX_ROWS))
df.printSchema()
df.show(5, truncate=False)

## MERGE en Iceberg (Nessie + bucket silver)

In [ ]:
full_name = silver.merge_into_iceberg(spark, df, cfg["table"])
print("Merge listo en", full_name)

## Verificacion

In [ ]:
print("Filas en", full_name, "->", spark.table(full_name).count())
spark.sql(f"SELECT * FROM {full_name} LIMIT 5").show(truncate=False)